# Compare Confidence Thresholds: DINO+SAM2 vs OWLv2+SAM2

This minimal notebook samples 20 frames and visualizes the effect of detection confidence thresholds in a 7-column layout:

Original | DINO 0.12 | DINO 0.20 | DINO 0.25 | OWLv2 0.12 | OWLv2 0.20 | OWLv2 0.25

## 1. Imports and configuration

In [ ]:
from __future__ import annotations

import random
import sys
import traceback
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    torch = None
    DEVICE = "cpu"


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / "scripts" / "auto_label" / "auto_label_gui.py").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Could not find repo root. Start Jupyter from prompt_video_segmenter or set REPO_ROOT manually.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
NOTEBOOKS_DIR = REPO_ROOT / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

FRAME_DIR = REPO_ROOT / "data" / "auto_label_demo" / "frames"
OUTPUT_DIR = REPO_ROOT / "outputs" / "compare_dino_owlv2_thresholds_20frames"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
N_SAMPLES = 20
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

DINO_THRESHOLDS = [0.12, 0.20, 0.25]
OWLV2_THRESHOLDS = [0.12, 0.20, 0.25]

print(f"Repo root : {REPO_ROOT}")
print(f"Frame dir : {FRAME_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Device    : {DEVICE}")

## 2. Locate frames and sample 20 images

In [ ]:
def list_images(root: Path) -> list[Path]:
    if not root.exists():
        return []
    return sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)


all_frames = list_images(FRAME_DIR)
if len(all_frames) < N_SAMPLES:
    raise ValueError(f"Only found {len(all_frames)} images in {FRAME_DIR}; need {N_SAMPLES}.")

rng = random.Random(RANDOM_SEED)
sampled_frames = rng.sample(all_frames, N_SAMPLES)

print(f"Found {len(all_frames)} frames; sampled {len(sampled_frames)}.")
for i, path in enumerate(sampled_frames[:5]):
    print(f"{i:02d}: {path}")

## 3. Load text prompts from existing autolabel GUI

In [ ]:
# Copied from AutoLabelWindow._build_config_panel() in scripts/auto_label/auto_label_gui.py.
TEXT_PROMPTS = [
    "hand", "glove",
    "pot", "pan", "lid", "cookware", "tray", "kettle",
    "bowl", "plate", "cup", "glass", "bottle", "jar",
    "container", "box", "package", "bag", "carton", "can",
    "knife", "fork", "spoon", "spatula", "tongs", "ladle", "whisk", "peeler", "scissors", "cutting board",
    "pasta", "noodles", "rice", "bread", "vegetable", "fruit", "meat", "fish", "egg", "cheese",
    "ingredient", "food", "dry food", "liquid", "water", "milk", "sauce", "oil", "powder", "sugar", "salt",
    "sink", "faucet", "stove", "cooktop", "oven", "microwave", "fridge",
    "drawer", "cabinet", "countertop", "table", "rack", "sponge", "towel",
]

print(f"Loaded {len(TEXT_PROMPTS)} prompts:")
print(", ".join(TEXT_PROMPTS))

## 4. Load DINO + SAM2 models for each threshold

In [ ]:
from scripts.auto_label.generate_mask_proposals import GroundingDINOSAM2Backend

dino_backends = {}
dino_load_errors = {}

for threshold in DINO_THRESHOLDS:
    try:
        print(f"Loading DINO+SAM2 threshold={threshold:.2f}")
        dino_backends[threshold] = GroundingDINOSAM2Backend(
            confidence_threshold=threshold,
            box_threshold=threshold,
            text_threshold=threshold,
            device=DEVICE,
        )
        dino_load_errors[threshold] = ""
    except Exception as exc:
        dino_load_errors[threshold] = f"{type(exc).__name__}: {exc}"
        print(f"DINO+SAM2 threshold={threshold:.2f} failed: {dino_load_errors[threshold]}")
        traceback.print_exc(limit=2)

## 5. Load OWLv2 + SAM2

In [ ]:
import owlv2_sam2_adapter_for_compare as owlv2_adapter

owlv2_state = None
owlv2_load_error = ""
try:
    owlv2_state = owlv2_adapter.load_sam3_model(DEVICE)
    print(f"OWLv2 loaded. SAM2 available={owlv2_state['sam2_available']} {owlv2_state['sam2_error']}")
except Exception as exc:
    owlv2_load_error = f"{type(exc).__name__}: {exc}"
    print(f"OWLv2+SAM2 failed to load: {owlv2_load_error}")
    traceback.print_exc(limit=2)

## 6. Run inference on sampled frames

In [ ]:
def run_dino_threshold(image_path: Path, threshold: float) -> dict:
    if threshold not in dino_backends:
        return {"instances": [], "error": dino_load_errors.get(threshold, "DINO backend missing")}
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        return {"instances": [], "error": f"Could not read image: {image_path}"}
    try:
        instances = dino_backends[threshold].generate(image_bgr, TEXT_PROMPTS, image_path.stem)
        return {"instances": instances, "error": ""}
    except Exception as exc:
        return {"instances": [], "error": f"{type(exc).__name__}: {exc}"}


def run_owlv2_threshold(image_path: Path, threshold: float) -> dict:
    if owlv2_state is None:
        return {"instances": [], "error": owlv2_load_error or "OWLv2 state missing"}
    try:
        instances = owlv2_adapter.predict_owlv2_sam2(owlv2_state, image_path, TEXT_PROMPTS, score_threshold=threshold)
        return {"instances": instances, "error": ""}
    except Exception as exc:
        return {"instances": [], "error": f"{type(exc).__name__}: {exc}"}


results = []
for idx, image_path in enumerate(sampled_frames):
    print(f"[{idx + 1:02d}/{len(sampled_frames)}] {image_path}")
    row = {"frame_path": image_path, "dino": {}, "owlv2": {}}
    for threshold in DINO_THRESHOLDS:
        row["dino"][threshold] = run_dino_threshold(image_path, threshold)
        print(f"  DINO {threshold:.2f}: {len(row['dino'][threshold]['instances'])} instances")
    for threshold in OWLV2_THRESHOLDS:
        row["owlv2"][threshold] = run_owlv2_threshold(image_path, threshold)
        print(f"  OWLv2 {threshold:.2f}: {len(row['owlv2'][threshold]['instances'])} instances")
    results.append(row)

print("Done.")

## 7. Visualize 7-column comparisons

In [ ]:
def load_rgb(image_path: Path) -> np.ndarray:
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        raise ValueError(f"Could not read image: {image_path}")
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


def color_for_index(index: int) -> tuple[float, float, float]:
    cmap = plt.get_cmap("tab20")
    return tuple(cmap(index % 20)[:3])


def normalize_instances(instances: list[dict]) -> list[dict]:
    rows = []
    for inst in instances or []:
        rows.append({
            "label": inst.get("label", inst.get("class", inst.get("name", "object"))),
            "score": inst.get("confidence", inst.get("score", None)),
            "bbox": inst.get("bbox_xyxy", inst.get("bbox", None)),
            "mask": inst.get("mask", None),
        })
    return rows


def draw_result(ax, image_rgb: np.ndarray, instances: list[dict], title: str, error: str = "") -> None:
    ax.imshow(image_rgb)
    ax.set_title(title, fontsize=9)
    ax.axis("off")
    if error:
        ax.text(8, 22, error[:80], color="white", fontsize=7, bbox={"facecolor": "red", "alpha": 0.7})
        return

    for i, inst in enumerate(normalize_instances(instances)):
        color = color_for_index(i)
        mask = inst.get("mask")
        if mask is not None:
            mask_arr = np.asarray(mask).astype(bool)
            if mask_arr.shape[:2] == image_rgb.shape[:2]:
                overlay = np.zeros((*mask_arr.shape, 4), dtype=float)
                overlay[mask_arr, :3] = color
                overlay[mask_arr, 3] = 0.35
                ax.imshow(overlay)

        bbox = inst.get("bbox")
        if bbox is not None and len(bbox) >= 4:
            x1, y1, x2, y2 = [float(v) for v in bbox[:4]]
            rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, linewidth=1.2, edgecolor=color)
            ax.add_patch(rect)
            label = str(inst.get("label", "object"))
            score = inst.get("score")
            if score is not None:
                label = f"{label} {float(score):.2f}"
            ax.text(x1, max(0, y1 - 3), label, color="white", fontsize=6, bbox={"facecolor": color, "alpha": 0.8, "pad": 1})


comparison_paths = []
for idx, row in enumerate(results):
    image_rgb = load_rgb(row["frame_path"])
    fig, axes = plt.subplots(1, 7, figsize=(28, 4.8), constrained_layout=True)
    draw_result(axes[0], image_rgb, [], "Original")
    col = 1
    for threshold in DINO_THRESHOLDS:
        result = row["dino"][threshold]
        draw_result(axes[col], image_rgb, result["instances"], f"DINO {threshold:.2f}\nN={len(result['instances'])}", result.get("error", ""))
        col += 1
    for threshold in OWLV2_THRESHOLDS:
        result = row["owlv2"][threshold]
        draw_result(axes[col], image_rgb, result["instances"], f"OWLv2 {threshold:.2f}\nN={len(result['instances'])}", result.get("error", ""))
        col += 1

    out_path = OUTPUT_DIR / f"threshold_comparison_{idx:03d}.png"
    fig.savefig(out_path, dpi=160)
    comparison_paths.append(out_path)
    plt.show()

print(f"Saved {len(comparison_paths)} figures to {OUTPUT_DIR}")

## 8. Save summary CSV

In [ ]:
summary_rows = []
for row, out_path in zip(results, comparison_paths):
    record = {
        "frame_path": str(row["frame_path"]),
        "comparison_output_path": str(out_path),
    }
    for threshold in DINO_THRESHOLDS:
        result = row["dino"][threshold]
        record[f"dino_{threshold:.2f}_num_instances"] = len(result["instances"])
        record[f"dino_{threshold:.2f}_error"] = result.get("error", "")
    for threshold in OWLV2_THRESHOLDS:
        result = row["owlv2"][threshold]
        record[f"owlv2_{threshold:.2f}_num_instances"] = len(result["instances"])
        record[f"owlv2_{threshold:.2f}_error"] = result.get("error", "")
    summary_rows.append(record)

summary_df = pd.DataFrame(summary_rows)
summary_csv = OUTPUT_DIR / "threshold_summary.csv"
summary_df.to_csv(summary_csv, index=False)
display(summary_df)
print(f"Saved: {summary_csv}")